# Lecture 06 exercises — Clustering

Run all cells. Derive answers in Markdown; TODOs are safe extension points.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "data" / "customer_segments.csv").exists():
            return path
    raise FileNotFoundError("Could not find data/customer_segments.csv")


ROOT = repo_root()
customers = pd.read_csv(ROOT / "data" / "customer_segments.csv")
features = [
    "recency_days",
    "orders_12m",
    "average_order_value",
    "annual_spend",
    "web_share",
    "tenure_months",
    "support_contacts_12m",
]
X = customers[features]
truth = customers["segment_truth"]  # Keep out of every fit and selection step.
display(X.describe().round(2))
scaler = StandardScaler()
Z = scaler.fit_transform(X)
print("means:", np.round(Z.mean(axis=0), 3), "stds:", np.round(Z.std(axis=0), 3))

## Exercise 1 — Centroid update

Expand and differentiate the fixed-membership objective. Discuss one-point and empty clusters.

In [ ]:
sample = np.array([-1.0, 0.5, 2.0, 4.0])
grid = np.linspace(-2, 5, 100)
loss = [np.sum((sample - center) ** 2) for center in grid]
print("mean:", sample.mean(), "grid minimizer:", grid[np.argmin(loss)])
# TODO: show your derivation.

## Exercise 2 — Scaling

Compare raw and standardized K-means at k=4. Which features dominate raw distance? Why is scaling a metric decision?

In [ ]:
raw = KMeans(n_clusters=4, n_init=20, random_state=606, algorithm="lloyd").fit(X)
scaled = KMeans(n_clusters=4, n_init=20, random_state=606, algorithm="lloyd").fit(Z)
print("raw inertia:", raw.inertia_, "scaled inertia:", scaled.inertia_)
print("standard deviations:", X.std().round(2).to_dict())
# TODO: compare profiles in original units.

## Exercise 3 — PCA via SVD

Compare leading singular vector and PCA loading up to sign; verify eigenvalue equals singular value squared over n-1.

In [ ]:
centered = Z - Z.mean(axis=0)
_, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
pca_check = PCA(n_components=2, svd_solver="full").fit(centered)
print("absolute alignment:", abs(float(Vt[0] @ pca_check.components_[0])))
print("SVD variance:", singular_values[0] ** 2 / (len(Z) - 1))
print("PCA variance:", pca_check.explained_variance_[0])

## Exercise 4 — k and stability

Inspect k=2..8 internal diagnostics, freeze a candidate, compare a second start with ARI, then report truth ARI only as post-hoc.

In [ ]:
rows, fitted = [], {}
for k in range(2, 9):
    estimator = KMeans(
        n_clusters=k,
        init="k-means++",
        n_init=20,
        max_iter=300,
        random_state=606,
        algorithm="lloyd",
    )
    labels = estimator.fit_predict(Z)
    fitted[k] = estimator
    rows.append(
        {
            "k": k,
            "inertia": estimator.inertia_,
            "silhouette": silhouette_score(Z, labels),
        }
    )
diagnostics = pd.DataFrame(rows).set_index("k")
display(diagnostics.round(3))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
diagnostics["inertia"].plot(marker="o", ax=axes[0])
axes[0].set(xlabel="k", ylabel="within-cluster sum of squares", title="Inertia")
diagnostics["silhouette"].plot(marker="o", ax=axes[1])
axes[1].set(xlabel="k", ylabel="mean silhouette", title="Internal diagnostic")
plt.tight_layout()

In [ ]:
candidate = int(diagnostics["silhouette"].idxmax())
print("candidate from silhouette:", candidate)
print("post-hoc synthetic ARI:", adjusted_rand_score(truth, fitted[candidate].labels_))
# TODO: compare a separate initialization at this k.